# What a VCF holds: summarize, filter, normalize

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fbcouto/multimerge-genomics-research/blob/main/notebooks/variants/summarize_filter_normalize.ipynb)

**The problem.** A variant caller hands you a VCF. Before any analysis, know what is in it: how many SNVs
and indels, how many passed the caller's filters, and the transition/transversion ratio (Ts/Tv), a quick
quality signal for SNVs. Then keep the confident calls with an explicit, reportable expression, and
split the multiallelic records, because many tools expect one alternative allele per record.

**What you need.** A VCF (plain or `.gz`, any size, one or many samples).

**What you get.** The summary numbers; how many records each candidate filter keeps; a filtered VCF; a
normalized VCF (one ALT per record, alleles trimmed), sorted, compressed and indexed.

In [ ]:
# Setup. Runs on Google Colab or on your own computer: on Colab, the first run installs MMGEN from the GitHub
# release (about a minute). Tables use pandas and plots matplotlib when installed (Colab has both); without
# them the notebook prints plain text and skips the plots.
import platform, subprocess, sys
from pathlib import Path

RELEASE = "https://github.com/fbcouto/multimerge-genomics-research/releases/download/v2.3.0"
try:
    import multimerge_genomics as mmg
except ImportError:
    if platform.system() != "Linux":
        print("MMGEN is not installed in this Python. Download the wheel for your system from\n"
              "  https://github.com/fbcouto/multimerge-genomics-research/releases/tag/v2.3.0\n"
              "install it with:  python -m pip install <the .whl file>\n"
              "and run this cell again.")
        raise
    print("Installing MMGEN (about a minute)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{RELEASE}/multimerge_genomics-2.3.0-cp310-abi3-manylinux_2_28_x86_64.whl"], check=True)
    import multimerge_genomics as mmg

IN_NOTEBOOK = "ipykernel" in sys.modules
try:
    from IPython.display import display, Markdown
except ImportError:                        # plain Python: print instead
    display, Markdown = print, str
try:
    import pandas as pd
    pd.set_option("display.width", 160)
    pd.set_option("display.max_columns", 30)
except ImportError:
    pd = None
try:
    import matplotlib
    if not IN_NOTEBOOK:
        matplotlib.use("Agg")              # plain Python: save the figures, open no window
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

# Chart colors (a palette checked for color-blind readers): two series, a two-sided scale, neutral ink
BLUE, ORANGE, RED, GRAY, INK = "#2a78d6", "#eb6834", "#e34948", "#a3a29c", "#52514e"


def show(rows, n=25):
    """Shows a list of dictionaries as a table (pandas when available, plain text otherwise)."""
    rows = list(rows)
    if pd is not None:
        display(pd.DataFrame(rows[:n]))
    else:
        for row in rows[:n]:
            print("  ".join(f"{k}: {v}" for k, v in row.items()))
    if len(rows) > n:
        print(f"... and {len(rows) - n} more rows")


def tidy(ax):
    """Recessive axes: no top or right frame, a light grid behind the data."""
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="#e4e3df", linewidth=0.8)
    ax.set_axisbelow(True)


def save_figure(fig, path):
    """Saves a figure; shows it in a notebook, closes it in plain Python."""
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print("Figure saved to", path)
    if IN_NOTEBOOK:
        plt.show()
    else:
        plt.close(fig)


print(mmg.engine_info())

## Data

By default, from the human demo pack (downloaded once, about 45 MB): the Genome in a Bottle (NIST) benchmark
calls v4.2.1 of HG002, a public genome of a volunteer who consented to open sharing, cut to chromosome 22
and the genes of some clinical panels. To use your own file, set `MY_VCF` and write your filter in
`EXPRESSION`.

In [ ]:
# Your file and filter (None = the demo data)
MY_VCF = None
# The records to keep. INFO/difficultregion is the Genome in a Bottle flag for hard-to-sequence regions;
# for your own files use your caller's fields, e.g. 'FILTER == "PASS" && QUAL >= 30 && INFO/DP >= 10'
EXPRESSION = 'FILTER == "PASS" && !(INFO/difficultregion)'
REFERENCE_FASTA = None       # the genome FASTA of the build: also left-aligns indels in step 4

import urllib.request, zipfile

DATA = Path("data")                          # the demo files are extracted here
OUT = Path("results") / "summarize_filter_normalize"         # everything this notebook writes
OUT.mkdir(parents=True, exist_ok=True)


def demo_file(pack, member):
    """One file of a demo pack: downloads the pack once into the working folder and extracts only that file."""
    target = DATA / member
    if not target.exists():
        if not Path(pack).exists():
            print(f"Downloading {pack} from the MMGEN release...")
            urllib.request.urlretrieve(f"{RELEASE}/{pack}", pack + ".part")
            Path(pack + ".part").replace(pack)
        with zipfile.ZipFile(pack) as z:
            z.extract(member, DATA)
    return str(target)


USING_DEMO = MY_VCF is None
VCF = MY_VCF or demo_file("mmgen_demo_human_v3.zip", "human/giab/HG002.vcf.gz")
print("VCF:", VCF)

## Step 1. What the file holds

`vcf_stats` reads the file once. A site counts once for each type among its ALT alleles. For human germline
SNVs, Ts/Tv is about 2.0-2.1 over a whole genome and near 3 in coding exons: random errors push it towards
0.5, so a low value warns of false positives. (This demo is chromosome 22 plus gene regions, both rich in
GC and coding sequence, so its Ts/Tv is higher than a whole genome's.)

In [ ]:
stats = {"original": mmg.vcf_stats(VCF)}
s = stats["original"]
print(s["text"])
print(f"Ts/Tv {s['ts_tv']:.2f}: " + ("in the usual range for human germline SNVs" if 1.8 <= s["ts_tv"] <= 3.3
                                    else "outside the usual range for human germline SNVs: look at the calls (or this is not germline data)"))

## Step 2. Filter with an expression

The expression language follows bcftools: fields `QUAL`, `POS`, `N_ALT`, `CHROM`, `ID`, `REF`, `ALT`,
`FILTER`, `TYPE` and `INFO/KEY` (alone, `INFO/KEY` tests that the flag is present); operators `== != < <= > >=`,
`~` and `!~` (for `TYPE` and `FILTER`), `&&`, `||`, `!` and parentheses. `TYPE == "snp"` keeps a site when
every ALT allele is a SNV, `TYPE ~ "snp"` when at least one is. A comparison with a missing value is false.
First see how many records some candidate filters keep; then the chosen `EXPRESSION` is written.

Compare the summary before and after. Removing hard-to-sequence regions usually removes a share of the
indels and some SNVs; a filter that changes Ts/Tv a lot was selecting on something related to errors.

In [ ]:
candidates = {
    "passed the caller's filters": 'FILTER == "PASS"',
    "SNVs only": 'TYPE == "snp"',
    "sites with an indel": 'TYPE ~ "indel"',
    "multiallelic sites": "N_ALT > 1",
    "seen by 3+ sequencing platforms (GIAB field)": "INFO/platforms >= 3",
    "in GIAB difficult regions (GIAB flag)": "INFO/difficultregion",
    "EXPRESSION (the chosen filter)": EXPRESSION,
}
rows = []
for what, expression in candidates.items():
    r = mmg.vcf_filter(VCF, str(OUT / "candidate.vcf.gz"), expression=expression)
    rows.append({"keeps": what, "expression": expression, "records": r["kept"], "of": r["read"]})
show(rows)

FILTERED = str(OUT / "filtered.vcf.gz")
kept = mmg.vcf_filter(VCF, FILTERED, expression=EXPRESSION)
print(f"Kept {kept['kept']:,} of {kept['read']:,} records -> {FILTERED}")

try:                                  # a mistake in an expression is pointed at, character by character
    mmg.vcf_filter(VCF, str(OUT / "candidate.vcf.gz"), expression="QUAL >= && TYPE")
except Exception as err:
    print("\nExample of an error message:\n" + "\n".join(str(err).splitlines()[:3]))

stats["filtered"] = mmg.vcf_stats(FILTERED)
show({"file": name, "records": s["records"], "SNV": s["snps"], "insertions": s["insertions"], "deletions": s["deletions"],
      "multiallelic": s["multiallelic"], "PASS": s["pass"], "Ts/Tv": round(s["ts_tv"], 3)} for name, s in stats.items())

## Step 3. Normalize: split multiallelic records and trim

A record with two ALT alleles becomes two records; `INFO` and `FORMAT` fields declared per allele
(`Number=A`, `R`, `G`) are split with it, and a genotype allele that belongs to the other record is written
as missing (`1/2` becomes `1/.` and `./1`). Every allele is then trimmed to its shortest form (`CA -> TA`
becomes `C -> T`, which is why some MNPs turn into SNVs). Left-alignment of indels in repeats needs the
genome (`REFERENCE_FASTA`). The output is indexed, ready for other tools.

In [ ]:
NORMALIZED = str(OUT / "normalized.vcf.gz")
norm = mmg.vcf_normalize(FILTERED, NORMALIZED, split_multiallelic=True, reference=REFERENCE_FASTA)
print(f"{norm['read']:,} records -> {norm['written']:,}: {norm['split']:,} multiallelic split, {norm['trimmed']:,} trimmed, "
      f"{norm['left_aligned']:,} left-aligned, {norm['position_changed']:,} moved" + (", re-sorted" if norm["resorted"] else ""))
mmg.index_file(NORMALIZED)

# One multiallelic record, before and after
mmg.vcf_filter(FILTERED, str(OUT / "multiallelic.vcf.gz"), expression="N_ALT > 1")
import gzip
example = next((l for l in gzip.open(OUT / "multiallelic.vcf.gz", "rt") if not l.startswith("#")), None)
if example:
    f = example.rstrip("\n").split("\t")
    print(f"\nBefore: {f[0]} {f[1]} {f[3]} -> {f[4]}   genotype {f[9].split(':')[0] if len(f) > 9 else '-'}")
    for line in mmg.query_region(NORMALIZED, f"{f[0]}:{f[1]}-{int(f[1]) + len(f[3]) - 1}", header=False):
        g = line.split("\t")
        if g[1] == f[1]:
            print(f"After:  {g[0]} {g[1]} {g[3]} -> {g[4]}   genotype {g[9].split(':')[0] if len(g) > 9 else '-'}")
stats["normalized"] = mmg.vcf_stats(NORMALIZED)
print("\n" + stats["normalized"]["text"])

## Check the result

In [ ]:
def check(condition, text):
    print(("OK     " if condition else "CHECK  ") + text)


check(kept["skipped"] == 0 and stats["original"]["skipped"] == 0, "no malformed line was skipped")
check(0 < kept["kept"] <= kept["read"], f"the filter kept some records, not more than it read ({kept['kept']:,})")
check(stats["normalized"]["multiallelic"] == 0, "no multiallelic record is left")
check(norm["written"] >= norm["read"], "normalizing only adds records (by splitting)")
check(norm["ref_mismatch"] == 0, "REF matches the genome everywhere it was checked (needs REFERENCE_FASTA)")
v = mmg.validate_file(NORMALIZED)
check(v["valid"] and v["sorted"], "the normalized file is valid and sorted")
check(1.8 <= stats["normalized"]["ts_tv"] <= 3.3, f"Ts/Tv {stats['normalized']['ts_tv']:.2f} is in the usual human germline range")
if USING_DEMO:
    assert stats["original"]["multiallelic"] > 0 and norm["split"] == stats["filtered"]["multiallelic"]
    assert kept["kept"] < kept["read"]
    print("Demo expectations met.")

## Methods paragraph

A starting text for the methods section of a paper or report, filled in with the version and the parameters
this run used. Edit it to your study, and cite the methods MMGEN reproduces (see References).

In [ ]:
import textwrap
VERSION = mmg.engine_info()["version"]
o, n = stats["original"], stats["normalized"]
methods = f"""The call set ({o['records']:,} records; {o['snps']:,} SNV and {o['insertions'] + o['deletions']:,} indel sites;
Ts/Tv {o['ts_tv']:.2f}) was summarized with vcf_stats. Records were kept when they satisfied the expression
`{EXPRESSION}` (vcf_filter; {kept['kept']:,} kept). Multiallelic records were split and alleles trimmed to their
shortest representation with vcf_normalize ({'indels left-aligned against ' + str(REFERENCE_FASTA) if REFERENCE_FASTA else 'without left-alignment'}),
giving {n['records']:,} biallelic records (Ts/Tv {n['ts_tv']:.2f}). All steps used MMGEN version {VERSION}
(Couto F. MMGEN: MultiMerge Genomics (version {VERSION}). Zenodo. https://doi.org/10.5281/zenodo.20417492)."""
print(textwrap.fill(" ".join(methods.split()), 110))

## Adapt it to your data

* Write `EXPRESSION` with your caller's fields: GATK (`QD`, `FS`, `MQ`, `INFO/DP`), DeepVariant (`QUAL`, `FILTER`),
  a population file (`INFO/AF < 0.01` for rare variants).
* Keep only some samples or regions: `vcf_filter(..., samples=["S1", "S2"], regions=["chr1:1-5000000"])`.
* Give `REFERENCE_FASTA` (same build) so indels in repeats are left-aligned, as `bcftools norm -f` does.
* For a cohort, run the summary per sample after `vcf_filter(samples=[name])`, and compare Ts/Tv between samples.

## References

* Danecek P. et al. (2021) Twelve years of SAMtools and BCFtools. *GigaScience* 10:giab008 (expressions, `norm -m-`).
* Tan A., Abecasis G.R., Kang H.M. (2015) Unified representation of genetic variants. *Bioinformatics* 31:2202.
* DePristo M.A. et al. (2011) A framework for variation discovery and genotyping using next-generation DNA
  sequencing data. *Nature Genetics* 43:491 (Ts/Tv as a quality measure).
* Wagner J. et al. (2022) Benchmarking challenging small variants with linked and long reads. *Cell Genomics* 2:100128.